# Dynamic Autoscaling: How Many Hosts Does SURF Need?

*Use case 1 of 3, about 10 minutes.*

We twin the SURF-SARA production cluster at SURF, the Dutch national supercomputing center.
SURF-SARA contains 277 hosts, each with 128 GB of RAM and 16 processing cores running at maximum 2.1 GHz.
OpenDT replays one week of its real-world operation (October 2022) through the OpenDC simulator.

![Datacenter digital twinning](resources/images/digital_twinning.png)

*High-level overview of datacenter digital twinning; a human-in-the-loop (HITL) oversees the process (OpenDT paper,
Figure 1).*

Run the cells in order (Shift+Enter). For each question, write down a guess first; a solution shows only after setting
`show=True` in its cell.

In [ ]:
import pandas as pd
from resources import *

check_setup()

## 1. Validating OpenDT against SURF

The first day of the trace (6-7 October 2022): predictions vs. ground truth (grey), i.e., the power draw measured at SURF.
The lower the MAPE (Mean Absolute Percentage Error), the more accurate the prediction.

In [ ]:
DAY = TRACE_START + pd.Timedelta("24h")  # the first 24 hours of the trace

day1 = simulate(until=DAY, label="OpenDT")
plot_power(day1)
print(f"MAPE: {mape(day1):.1f}%")

In [ ]:
ask_question("1.1")

In [ ]:
give_solution("1.1", show=False)  # show=True shows the solution

## 2. What-If Analysis: Fewer and More Hosts

The same day with 1/4, 1/2, 1x, and 10x the hosts (69, 138, 277, and 2,770 hosts).

In [ ]:
ask_question("1.2")

In [ ]:
sizes = [69, 138, 277, 2770]
runs = simulate_many("hosts", sizes, until=DAY)

table = pd.DataFrame([run.summary() for run in runs.values()], index=list(runs))
table[["avg_power_kw", "jobs_waited", "mean_wait_min", "waiting_at_end"]].round(1)

*Columns:* `avg_power_kw`, the average power draw [kW]; `jobs_waited`, the jobs that cannot start right away
(all hosts busy); `mean_wait_min`, the average waiting time per job [minutes]; `waiting_at_end`, the jobs still
waiting at the end of the day.

In [ ]:
ask_question("1.3")

In [ ]:
give_solution("1.3", show=False)  # show=True shows the solution

In [ ]:
ask_question("1.4")

In [ ]:
give_solution("1.4", show=False)  # show=True shows the solution

In [ ]:
plot_power(*runs.values());  # optional: the four simulations in one plot

## 3. Hosts Needed over the Day

In the simulation, each running job occupies one whole host (16 cores): hosts in use = running jobs, every 150 s.
A dynamic autoscaler keeps these hosts on, plus a few spare ones, and switches off the rest.

In [ ]:
ask_question("1.5")

In [ ]:
HOSTS = 277  # hosts at SURF
in_use = day1.service["tasks_active"]  # busy hosts at each moment (one job per host)

ax = in_use.plot(label="Hosts in Use")
ax.fill_between(in_use.index, in_use, HOSTS, alpha=0.2, label="Idle Hosts")
ax.set(ylim=(0, 300), xlabel="Time [UTC]", ylabel="Hosts")
legend_above(ax)

In [ ]:
give_solution("1.5", show=False)  # show=True shows the solution

In [ ]:
SPARE = 10  # hosts kept on for new jobs
IDLE_POWER = 25  # watts an idle SURF host draws

needed = (in_use + SPARE).clip(upper=HOSTS)
switched_off = HOSTS - needed
saved_kwh = (switched_off * IDLE_POWER).mean() / 1000 * 24  # average power [kW] x 24 hours
used_kwh = day1.summary()["energy_kwh"]
print(f"Switching off idle hosts saves {saved_kwh:.0f} of {used_kwh:.0f} kWh ({saved_kwh / used_kwh:.0%}).")

In [ ]:
ask_question("1.6")

In [ ]:
give_solution("1.6", show=False)  # show=True shows the solution

In [ ]:
ask_question("1.7")

In [ ]:
give_solution("1.7", show=False)  # show=True shows the solution

In [ ]:
ask_question("1.8")

In [ ]:
low_idle = simulate(until=DAY, idle_power=10, label="OpenDT, 10 W idle")
plot_power(day1, low_idle)
print(f"Average power draw: {day1.summary()['avg_power_kw']:.1f} vs. {low_idle.summary()['avg_power_kw']:.1f} kW")
print(f"MAPE: {mape(day1):.1f}% vs. {mape(low_idle):.1f}%")

In [ ]:
give_solution("1.8", show=False)  # show=True shows the solution

## Bonus: Changing the Hosts in the Live Demo

*Optional.* In the live demo, we change the hosts from the REST API (`PUT /api/topology`); from then on, OpenDT twins
the datacenter with 2,770 hosts. The same view, from two simulations:

In [ ]:
SWITCH = TRACE_START + pd.Timedelta("10h")  # 08:00 UTC on 7 October

scaled = day1.power["power_draw"].copy()
scaled[SWITCH:] = runs[2770].power["power_draw"][SWITCH:]  # from 08:00, the simulation with 2,770 hosts
plot_power(scaled.rename("OpenDT, 2,770 hosts from 08:00"));

*Next:* [2.calibration.ipynb](2.calibration.ipynb), simulation calibration.